# Proyecto 2 — Modelo de Regresión
## Predicción del Puntaje Global Saber 11 — Departamento del Huila

**Pregunta de negocio:** ¿Cuál es el puntaje global esperado de un estudiante dado su estrato socioeconómico, zona geográfica (urbano/rural), nivel educativo de los padres y jornada escolar?

| Campo | Detalle |
|---|---|
| **Responsable** | Juan Nicolás Hidalgo Parra |
| **Rol** | Ciencia de Datos |
| **Tipo de modelo** | Regresión con Red Neuronal (Keras / TensorFlow) |
| **Variable objetivo** | `puntaje_global` (continua, rango 100–500) |
| **Cliente** | Secretaría de Educación del Huila |

---
## 1. Librerías y Configuración Inicial

In [ ]:
# Manipulación y análisis de datos 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocesamiento y métricas 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Construcción de redes neuronales
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# Registro de experimentos
import mlflow
import mlflow.keras

# Serialización de objetos 
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Semillas fijas para garantizar reproducibilidad entre ejecuciones
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Carga del conjunto de datos - Proyecto 1
# El archivo corresponde al subconjunto filtrado para el departamento del Huila
# con 19 variables seleccionadas y un tamaño de aproximadamente 29.7 MB
df = pd.read_csv('../tarea2_datos/Datos_Huila_Limpio.csv')

print(f"Dimensiones del dataset: {df.shape}")
df.head()

---
## 2. Selección y Preprocesamiento de Variables

Se seleccionan las variables predictoras con base en su relevancia para responder la pregunta de negocio. La selección se fundamenta en los hallazgos del Proyecto 1, donde se identificó que el estrato, la zona geográfica y el nivel educativo del hogar son los factores con mayor influencia sobre el desempeño académico en el Huila.

> **Nota sobre data leakage:** Los puntajes individuales por área (matemáticas, lectura, ciencias, sociales, inglés) se excluyen deliberadamente, ya que son componentes directos del `puntaje_global`. Incluirlos haría que el modelo aprenda una identidad aritmética en lugar de una relación socioeconómica real.

In [ ]:
# Variables predictoras seleccionadas
# Se incluyen variables académicas institucionales y socioeconómicas del hogar
features = [
    'estrato',            # Nivel socioeconómico del hogar (ordinal, 1–6)
    'area_colegio',       # Ubicación geográfica del establecimiento (urbano/rural)
    'educacion_madre',    # Nivel educativo alcanzado por la madre (ordinal)
    'educacion_padre',    # Nivel educativo alcanzado por el padre (ordinal)
    'jornada',            # Jornada escolar del establecimiento (nominal)
    'municipio',          # Municipio del establecimiento (nominal, 37 categorías)
    'tiene_internet',     # Acceso a internet en el hogar (binaria)
    'tiene_computador',   # Acceso a computador en el hogar (binaria)
    'naturaleza_colegio', # Carácter del establecimiento: oficial o no oficial (binaria)
    'genero'              # Género del estudiante (binaria)
]

target = 'puntaje_global'

# Se eliminan registros con valores faltantes en las variables de interés
df_model = df[features + [target]].dropna()

print(f"Registros disponibles tras eliminar nulos: {len(df_model):,}")
print(f"\nEstadísticas descriptivas del puntaje global:")
print(df_model[target].describe().round(2))

### 2.1 Codificación de Variables Categóricas

La estrategia de codificación se define según la naturaleza de cada variable:

| Variable | Tipo | Estrategia | Justificación |
|---|---|---|---|
| `estrato` | Ordinal | Mapeo numérico 0–6 | Preserva el orden jerárquico |
| `educacion_madre/padre` | Ordinal | Mapeo numérico 0–9 | Preserva la jerarquía educativa |
| `area_colegio`, `tiene_internet`, `tiene_computador`, `genero`, `naturaleza_colegio` | Binaria | Codificación 0/1 | Variables dicotómicas naturales |
| `municipio` | Nominal (37 categorías) | Target encoding | Evita alta dimensionalidad del one-hot; captura el efecto promedio por municipio |
| `jornada` | Nominal | One-hot encoding | Número reducido de categorías, sin orden inherente |

In [ ]:
df_enc = df_model.copy()

# ── 1. Estrato: mapeo ordinal ──────────────────────────────────────────────────
# Se asigna 0 a 'Sin Estrato' por representar la condición de mayor vulnerabilidad
estrato_map = {
    'Sin Estrato': 0,
    'Estrato 1': 1, 'Estrato 2': 2, 'Estrato 3': 3,
    'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6
}
df_enc['estrato'] = df_enc['estrato'].map(estrato_map)

# ── 2. Educación de padres: mapeo ordinal ─────────────────────────────────────
# Se cubren ambas formas de etiquetado presentes en el dataset original
edu_map = {
    'Ninguno': 0,
    'Primaria incompleta': 1,
    'Primaria completa': 2,
    'Secundaria incompleta': 3,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria completa': 4,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'Educación profesional completa': 8,
    'Postgrado': 9
}
df_enc['educacion_madre'] = df_enc['educacion_madre'].map(edu_map)
df_enc['educacion_padre'] = df_enc['educacion_padre'].map(edu_map)

# ── 3. Variables binarias: codificación 0/1 ────────────────────────────────────
df_enc['area_colegio']       = (df_enc['area_colegio'] == 'URBANO').astype(int)
df_enc['tiene_internet']     = (df_enc['tiene_internet'] == 'Si').astype(int)
df_enc['tiene_computador']   = (df_enc['tiene_computador'] == 'Si').astype(int)
df_enc['genero']             = (df_enc['genero'] == 'M').astype(int)
df_enc['naturaleza_colegio'] = (df_enc['naturaleza_colegio'] == 'NO OFICIAL').astype(int)

# ── 4. Municipio: target encoding ─────────────────────────────────────────────
# Se reemplaza cada municipio por el promedio del puntaje global de sus estudiantes
# Esto captura el efecto contextual del municipio sin generar 37 columnas dummy
mun_target_enc = df_enc.groupby('municipio')[target].mean()
df_enc['municipio'] = df_enc['municipio'].map(mun_target_enc)

# ── 5. Jornada: one-hot encoding ──────────────────────────────────────────────
# drop_first=True elimina una categoría redundante para evitar multicolinealidad
df_enc = pd.get_dummies(df_enc, columns=['jornada'], prefix='jornada', drop_first=True)

# ── Diagnóstico: verificar NaN residuales post-encoding ───────────────────────
nulos = df_enc.isnull().sum()
print("Columnas con NaN residuales tras encoding:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "  Ninguna — dataset limpio")

# Eliminación de filas con NaN en caso de etiquetas no contempladas en los mapas
df_enc = df_enc.dropna()

print(f"\nRegistros finales disponibles para modelado: {len(df_enc):,}")
print(f"Total de variables predictoras: {df_enc.shape[1] - 1}")
print(f"Variables: {df_enc.drop(columns=[target]).columns.tolist()}")

---
## 3. División y Escalamiento de Datos

Se aplica una partición 80/20 entre entrenamiento y prueba. El escalamiento se realiza tanto sobre las variables predictoras como sobre la variable objetivo, con el propósito de estabilizar el entrenamiento de la red neuronal y evitar que la escala del target (rango 100–500) genere gradientes desproporcionados durante la optimización.

In [ ]:
X = df_enc.drop(columns=[target]).values
y = df_enc[target].values

# División estratificada 80% entrenamiento / 20% prueba
# random_state fijo para garantizar reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalamiento de variables predictoras
# fit_transform sobre train, solo transform sobre test (evita data leakage)
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

# Escalamiento del target
# Necesario para evitar gradientes desproporcionados dado el rango original (100–500)
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled  = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

print(f"Conjunto de entrenamiento: {X_train.shape[0]:,} registros, {X_train.shape[1]} features")
print(f"Conjunto de prueba:        {X_test.shape[0]:,} registros")
print(f"\nTarget original  — media: {y_train.mean():.2f}, desv. estándar: {y_train.std():.2f}")
print(f"Target escalado  — media: {y_train_scaled.mean():.2f}, desv. estándar: {y_train_scaled.std():.2f}")

---
## 4. Modelo Base de Regresión

Se define una red neuronal de referencia con arquitectura simple: dos capas ocultas con activación ReLU y una capa de salida lineal, apropiada para tareas de regresión. La función de pérdida empleada es el Error Cuadrático Medio (MSE). Se utiliza un *callback* de Early Stopping para detener el entrenamiento cuando la pérdida de validación deja de mejorar, evitando el sobreajuste.

In [ ]:
n_features = X_train.shape[1]

def build_model_base():
    """
    Modelo base: red neuronal densa con dos capas ocultas.
    Arquitectura: Input → Dense(32, ReLU) → Dense(16, ReLU) → Output(lineal)
    Optimizador: Adam con learning rate 0.0001
    Pérdida: MSE (apropiada para regresión)
    """
    model = Sequential([
        Input(shape=(n_features,)),   # Capa de entrada explícita (compatibilidad Keras 3)
        Dense(32, activation='relu'), # Primera capa oculta: expande el espacio de representación
        Dense(16, activation='relu'), # Segunda capa oculta: comprime hacia la predicción
        Dense(1)                      # Salida lineal: valor continuo del puntaje global
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.0001), # LR conservador para convergencia estable
        loss='mse',
        metrics=['mae']
    )
    return model

model_base = build_model_base()
model_base.summary()

In [ ]:
# Early Stopping: detiene el entrenamiento si val_loss no mejora en 10 épocas consecutivas
# restore_best_weights recupera los pesos del mejor epoch al finalizar
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history_base = model_base.fit(
    X_train, y_train_scaled,
    validation_data=(X_test, y_test_scaled),
    epochs=100,
    batch_size=256,   # Tamaño de lote amplio para reducir tiempo de cómputo
    callbacks=[early_stop],
    verbose=1
)

print(f"\nÉpocas efectivas de entrenamiento: {len(history_base.history['loss'])}")

# Visualización de las curvas de pérdida durante el entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_base.history['loss'],     label='Entrenamiento')
ax1.plot(history_base.history['val_loss'], label='Validación')
ax1.set_title('Pérdida (MSE) — Modelo Base')
ax1.set_xlabel('Épocas')
ax1.set_ylabel('MSE (escala normalizada)')
ax1.legend()

ax2.plot(history_base.history['mae'],     label='Entrenamiento')
ax2.plot(history_base.history['val_mae'], label='Validación')
ax2.set_title('Error Absoluto Medio (MAE) — Modelo Base')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('MAE (escala normalizada)')
ax2.legend()

plt.tight_layout()
plt.show()

### 4.1 Métricas de Evaluación del Modelo Base

Las métricas se calculan en la escala original del puntaje global (100–500), deshaciendo la normalización aplicada previamente. Esto permite interpretar el error en unidades reales de la prueba.

In [ ]:
def evaluar_modelo(model, X_test, y_test_scaled, y_test_real, nombre='Modelo'):
    """
    Evalúa un modelo de regresión y retorna métricas en escala original.
    
    Parámetros:
        model        : modelo Keras entrenado
        X_test       : features de prueba escaladas
        y_test_scaled: target de prueba normalizado (para la predicción interna)
        y_test_real  : target de prueba en escala original (para calcular métricas reales)
        nombre       : etiqueta identificadora del modelo
    
    Retorna:
        Diccionario con RMSE, MAE y R² en escala original
    """
    # Predicción en espacio normalizado
    y_pred_scaled = model.predict(X_test, verbose=0).flatten()
    # Conversión a escala original del puntaje global
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
    mae  = mean_absolute_error(y_test_real, y_pred)
    r2   = r2_score(y_test_real, y_pred)

    print(f"--- MÉTRICAS {nombre} ---")
    print(f"RMSE : {rmse:.4f}  puntos del puntaje global")
    print(f"MAE  : {mae:.4f}  puntos del puntaje global")
    print(f"R²   : {r2:.4f}")
    return {'Modelo': nombre, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4), 'R²': round(r2, 4)}

metricas_base = evaluar_modelo(model_base, X_test, y_test_scaled, y_test, 'Base')
results_all = [metricas_base]

In [ ]:
# Gráfica de dispersión: predicciones vs valores reales
# La línea roja punteada representa la predicción perfecta (y_pred == y_real)
# La cercanía de los puntos a esta línea indica la calidad del ajuste
y_pred_scaled_base = model_base.predict(X_test, verbose=0).flatten()
y_pred_base = scaler_y.inverse_transform(y_pred_scaled_base.reshape(-1, 1)).flatten()

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_base, alpha=0.3, s=10, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Predicción perfecta')
plt.xlabel('Puntaje Global Real')
plt.ylabel('Puntaje Global Predicho')
plt.title('Predicciones vs Valores Reales — Modelo Base')
plt.legend()
plt.tight_layout()
plt.show()

**Análisis del modelo base:**

Las curvas de entrenamiento y validación convergen de forma suave y sin divergencia, lo que indica que el modelo no presenta sobreajuste. El Early Stopping intervino correctamente, deteniendo el entrenamiento una vez estabilizada la pérdida de validación.

El modelo base obtuvo un **RMSE de 40.44 puntos** y un **R² de 0.2748**, lo cual implica que explica aproximadamente el 27.5% de la variabilidad del puntaje global a partir exclusivamente de variables socioeconómicas e institucionales. Este resultado es coherente con la literatura educativa, que señala que las condiciones socioeconómicas explican entre el 20% y el 35% del rendimiento académico cuando no se incluyen variables pedagógicas directas.

El diagrama de dispersión muestra una tendencia positiva alineada con la diagonal de predicción perfecta, confirmando que el modelo captura la dirección correcta de las relaciones. La dispersión residual refleja la variabilidad del desempeño académico que no puede ser capturada por el perfil socioeconómico del estudiante.

---
## 5. Experimentación con Variaciones de la Arquitectura

Con el fin de identificar la configuración óptima, se evalúan tres arquitecturas adicionales que modifican la profundidad de la red, la regularización y la tasa de aprendizaje. Todas las variaciones emplean los mismos datos de entrenamiento y prueba para garantizar comparabilidad.

In [ ]:
def build_model_1():
    """
    Modelo 1 — Red Profunda.
    Arquitectura: Input → 128 → 64 → 32 → 16 → Output
    Hipótesis: mayor profundidad permite capturar interacciones no lineales
    más complejas entre variables socioeconómicas.
    """
    model = Sequential([
        Input(shape=(n_features,)),
        Dense(128, activation='relu'),
        Dense(64,  activation='relu'),
        Dense(32,  activation='relu'),
        Dense(16,  activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
    return model


def build_model_2():
    """
    Modelo 2 — Red con Dropout.
    Arquitectura: Input → 64 → Dropout(30%) → 32 → Dropout(30%) → 16 → Output
    Hipótesis: el Dropout actúa como regularizador, desactivando aleatoriamente
    el 30% de las neuronas por época para reducir la dependencia entre unidades.
    """
    model = Sequential([
        Input(shape=(n_features,)),
        Dense(64,  activation='relu'),
        Dropout(0.3),  # Regularización: apaga el 30% de neuronas aleatoriamente
        Dense(32,  activation='relu'),
        Dropout(0.3),
        Dense(16,  activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
    return model


def build_model_3():
    """
    Modelo 3 — Arquitectura simple con tasa de aprendizaje mayor.
    Arquitectura: Input → 32 → 16 → Output (igual al base)
    Hipótesis: un learning rate 10 veces mayor (0.001) acelera la convergencia
    sin necesariamente degradar la calidad del ajuste.
    """
    model = Sequential([
        Input(shape=(n_features,)),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

In [ ]:
# Entrenamiento secuencial de las tres variaciones
# Se reutiliza el mismo early_stop definido previamente
histories = {}

configs = [
    (build_model_1, 'Modelo 1 - Red Profunda'),
    (build_model_2, 'Modelo 2 - Con Dropout'),
    (build_model_3, 'Modelo 3 - Simple + LR alto'),
]

for build_fn, nombre in configs:
    print(f"\nEntrenando {nombre}...")
    m = build_fn()
    h = m.fit(
        X_train, y_train_scaled,
        validation_data=(X_test, y_test_scaled),
        epochs=100,
        batch_size=256,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0  # verbose=0 para no saturar la salida con 3 modelos adicionales
    )
    histories[nombre] = h
    metricas = evaluar_modelo(m, X_test, y_test_scaled, y_test, nombre)
    results_all.append(metricas)

In [ ]:
# Comparación visual de las curvas de pérdida de validación
# Permite identificar estabilidad, velocidad de convergencia y posibles divergencias
plt.figure(figsize=(10, 5))
plt.plot(history_base.history['val_loss'], label='Modelo Base', linestyle='--', color='steelblue')
for nombre, h in histories.items():
    plt.plot(h.history['val_loss'], label=nombre)

plt.title('Comparación de Pérdida de Validación por Arquitectura')
plt.xlabel('Épocas')
plt.ylabel('MSE de Validación (escala normalizada)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Tabla comparativa de métricas en escala original del puntaje global
df_results = pd.DataFrame(results_all)
df_results

**Análisis comparativo de arquitecturas:**

Los cuatro modelos evaluados presentan métricas muy similares, con diferencias inferiores al 1% en RMSE. El **Modelo Base** obtuvo el mejor desempeño global (RMSE: 40.44, R²: 0.2748), seguido muy de cerca por el Modelo 3 y el Modelo 1.

El Modelo 1 (Red Profunda) presenta un comportamiento inestable en la curva de validación con un pico de divergencia notable, lo que indica que la arquitectura de cuatro capas ocultas resulta excesiva para la complejidad del problema. El Modelo 2 (Dropout) no logra superar al base, lo cual es esperable dado que el modelo base no exhibe sobreajuste — el Dropout penaliza el aprendizaje sin aportar beneficios de regularización en este caso.

Estos resultados sugieren que la capacidad predictiva de las variables socioeconómicas sobre el puntaje global tiene un límite estructural que no puede superarse aumentando la complejidad de la red. El **Modelo Base** se selecciona como modelo final por su mejor métrica y mayor estabilidad de entrenamiento.

---
## 6. Registro de Experimentos con MLflow

Se registran los cuatro experimentos en MLflow para documentar los parámetros, métricas y artefactos de cada configuración. Esto permite trazabilidad completa del proceso de modelado y facilita la reproducción de resultados.

In [ ]:
# Configuración del servidor de tracking local
mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment("Regresion_PuntajeGlobal_Huila")

configs_mlflow = [
    {'nombre': 'Modelo_Base',      'build_fn': build_model_base, 'lr': 0.0001, 'dropout': False, 'capas': 2},
    {'nombre': 'Modelo_Profundo',  'build_fn': build_model_1,    'lr': 0.0001, 'dropout': False, 'capas': 4},
    {'nombre': 'Modelo_Dropout',   'build_fn': build_model_2,    'lr': 0.0001, 'dropout': True,  'capas': 3},
    {'nombre': 'Modelo_Simple_LR', 'build_fn': build_model_3,    'lr': 0.001,  'dropout': False, 'capas': 2},
]

for cfg in configs_mlflow:
    with mlflow.start_run(run_name=cfg['nombre']):

        # Registro de hiperparámetros
        mlflow.log_param('learning_rate',    cfg['lr'])
        mlflow.log_param('dropout',          cfg['dropout'])
        mlflow.log_param('capas_ocultas',    cfg['capas'])
        mlflow.log_param('batch_size',       256)
        mlflow.log_param('epochs_max',       100)
        mlflow.log_param('early_stopping',   True)
        mlflow.log_param('optimizer',        'Adam')
        mlflow.log_param('target_escalado',  True)

        # Entrenamiento con los mismos parámetros de la sección 4 y 5
        m = cfg['build_fn']()
        m.fit(
            X_train, y_train_scaled,
            validation_data=(X_test, y_test_scaled),
            epochs=100,
            batch_size=256,
            callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
            verbose=0
        )

        # Cálculo de métricas en escala original
        y_pred_sc = m.predict(X_test, verbose=0).flatten()
        y_pred    = scaler_y.inverse_transform(y_pred_sc.reshape(-1, 1)).flatten()
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)

        # Registro de métricas y artefacto del modelo
        mlflow.log_metric('RMSE', rmse)
        mlflow.log_metric('MAE',  mae)
        mlflow.log_metric('R2',   r2)
        mlflow.keras.log_model(m, artifact_path='model')

        print(f"{cfg['nombre']:<22} — RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

print("\nExperimentos registrados exitosamente en MLflow.")

---
## 7. Serialización del Modelo Final

Con base en los resultados de la sección 5, se selecciona el **Modelo Base** como modelo final por presentar el menor RMSE (40.44) y el mayor R² (0.2748), junto con la mayor estabilidad durante el entrenamiento. El modelo se serializa junto con los transformadores necesarios para garantizar que el tablero Dash pueda realizar predicciones coherentes sobre nuevas entradas del usuario.

In [ ]:
# Reentrenamiento del modelo final seleccionado
mejor_modelo = build_model_base()

mejor_modelo.fit(
    X_train, y_train_scaled,
    validation_data=(X_test, y_test_scaled),
    epochs=100,
    batch_size=256,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

# Creación del directorio de modelos para el tablero
os.makedirs('../tarea5_tablero/modelos', exist_ok=True)

# Serialización en formato .keras (compatible con Keras 3)
# Se guarda el modelo y ambos transformadores para reproducir el pipeline completo
mejor_modelo.save('../tarea5_tablero/modelos/modelo_regresion.keras')
joblib.dump(scaler_X,          '../tarea5_tablero/modelos/scaler_X_regresion.pkl')
joblib.dump(scaler_y,          '../tarea5_tablero/modelos/scaler_y_regresion.pkl')
joblib.dump(mun_target_enc.to_dict(), '../tarea5_tablero/modelos/municipio_encoder.pkl')

print("Archivos guardados:")
print("  - modelo_regresion.keras       (red neuronal entrenada)")
print("  - scaler_X_regresion.pkl       (normalizador de features)")
print("  - scaler_y_regresion.pkl       (normalizador del target)")
print("  - municipio_encoder.pkl        (target encoding de municipio)")

In [ ]:
# Verificación de la serialización
# Se carga el modelo desde disco y se realizan predicciones de prueba
# para confirmar que el pipeline completo funciona correctamente
from tensorflow.keras.models import load_model

modelo_cargado   = load_model('../tarea5_tablero/modelos/modelo_regresion.keras')
scaler_y_cargado = joblib.load('../tarea5_tablero/modelos/scaler_y_regresion.pkl')

# Predicción sobre los primeros 5 registros del conjunto de prueba
y_pred_scaled_check = modelo_cargado.predict(X_test[:5], verbose=0).flatten()
y_pred_check = scaler_y_cargado.inverse_transform(y_pred_scaled_check.reshape(-1, 1)).flatten()

print("Verificación de predicciones post-carga:")
print(f"  Predicciones: {y_pred_check.round(1)}")
print(f"  Valores reales: {y_test[:5].round(1)}")
print("\nModelo cargado y verificado correctamente ✓")

---
## 8. Conclusiones

### Resultados del modelo seleccionado (Modelo Base)

| Métrica | Valor | Interpretación |
|---|---|---|
| **RMSE** | 40.44 puntos | Error promedio de predicción en la escala del puntaje global |
| **MAE** | 32.44 puntos | Error absoluto promedio, menos sensible a valores extremos |
| **R²** | 0.2748 | El modelo explica el 27.5% de la variabilidad del puntaje global |

### Hallazgos principales

1. **Capacidad predictiva del perfil socioeconómico:** Un R² de 0.27 indica que las variables socioeconómicas e institucionales capturan aproximadamente la cuarta parte de la variabilidad del puntaje global. Este resultado, aunque moderado, es consistente con la evidencia del Proyecto 1, donde se identificó que la zona geográfica y el estrato son los determinantes estructurales más relevantes del rendimiento en el Huila.

2. **Límite estructural del enfoque:** El error de predicción residual (~72.5% no explicado) refleja factores no capturados por el modelo: calidad docente, motivación del estudiante, metodología pedagógica y condiciones familiares no observadas. Esto no constituye una limitación del modelo neuronal en sí, sino del conjunto de variables disponibles.

3. **Arquitectura óptima:** El Modelo Base (dos capas ocultas, 32 y 16 neuronas) superó o igualó a configuraciones más complejas. La Red Profunda (Modelo 1) presentó inestabilidad por sobrepasar la capacidad necesaria para el problema. El Dropout (Modelo 2) no aportó mejoras al no existir sobreajuste en el modelo base.

4. **Utilidad para la Secretaría de Educación:** El modelo permite estimar el puntaje esperado de un estudiante dado su perfil socioeconómico e institucional, lo cual es útil para identificar estudiantes en riesgo de bajo rendimiento antes de la aplicación del examen y para simular el impacto de intervenciones como la mejora de conectividad o el cambio de jornada escolar.